# <span style="color:red">HW2_Reza_Zamani</span>

# <span style="color:green">This project is for Python for MFE program at UC Berkeley </span>

## Libraries 

In [11]:
## installing some libraries 
!pip install papermill

In [12]:
import sqlite3
import traceback
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import papermill as pm
from tqdm import tqdm
import matplotlib.pyplot as plt

In [13]:

# ----------------------------
# Config
# ----------------------------
DB_PATH = "data3_clean.db"
TABLE = "ohlc_hw_cleaned"
TEMPLATE_PATH = "02_stock_report_template.ipynb"
KERNEL_NAME = "python3"  # set to None to let papermill use notebook metadata


@dataclass(frozen=True)
class DateWindow:
    start: pd.Timestamp
    end: pd.Timestamp

    @property
    def start_str(self) -> str:
        return self.start.strftime("%Y-%m-%d")

    @property
    def end_str(self) -> str:
        return self.end.strftime("%Y-%m-%d")



In [14]:

# ----------------------------
# Helpers
# ----------------------------
def get_tickers(conn: sqlite3.Connection) -> list[str]:
    q = f"SELECT DISTINCT ticker FROM {TABLE} ORDER BY ticker"
    s = pd.read_sql_query(q, conn)["ticker"]
    return s.dropna().astype(str).tolist()


def get_last_6_complete_months(conn: sqlite3.Connection) -> DateWindow:
    q = f"SELECT MAX(ts) as max_date FROM {TABLE}"
    max_date_raw = pd.read_sql_query(q, conn)["max_date"].iloc[0]

    if pd.isna(max_date_raw):
        raise ValueError("Database table has no dates (MAX(ts) is NULL).")

    max_date = pd.to_datetime(max_date_raw)

    # End date = last day of the last fully completed month relative to max_date
    # Example: if max_date is 2024-07-10, last complete month ends 2024-06-30
    end_date = (max_date.to_period("M").start_time - pd.Timedelta(days=1)).normalize()

    # Start date = first day of the month 5 months before end month (inclusive gives 6 full months)
    # Example: end month June -> start month January 1
    start_month_start = (end_date.to_period("M").start_time - pd.DateOffset(months=5)).normalize()

    return DateWindow(start=start_month_start, end=end_date)


def ensure_output_dir() -> Path:
    today = datetime.now().strftime("%Y-%m-%d")
    out = Path("reports") / today
    out.mkdir(parents=True, exist_ok=True)
    return out


def run_reports(
    tickers: list[str],
    window: DateWindow,
    output_dir: Path,
    template_path: str,
) -> tuple[list[str], list[tuple[str, str]]]:
    successful: list[str] = []
    failed: list[tuple[str, str]] = []

    log_path = output_dir / "papermill_failures.log"

    for ticker in tqdm(tickers, desc="Generating reports"):
        output_path = output_dir / f"{ticker}_report.ipynb"
        try:
            pm.execute_notebook(
                input_path=template_path,
                output_path=str(output_path),
                parameters={
                    "ticker": ticker,
                    "start_date": window.start_str,
                    "end_date": window.end_str,
                },
                kernel_name=KERNEL_NAME if KERNEL_NAME else None,
            )
            successful.append(ticker)

        except Exception:
            tb = traceback.format_exc()
            failed.append((ticker, tb))

            # Append full traceback to log file for debugging
            with open(log_path, "a", encoding="utf-8") as f:
                f.write(f"\n{'='*80}\nTICKER: {ticker}\n{tb}\n")

    return successful, failed


def compute_metrics_for_ticker(conn: sqlite3.Connection, ticker: str, window: DateWindow) -> dict | None:
    q = f"""
        SELECT ts, close
        FROM {TABLE}
        WHERE ticker = ?
          AND ts BETWEEN ? AND ?
        ORDER BY ts
    """
    df = pd.read_sql_query(q, conn, params=(ticker, window.start_str, window.end_str))

    if df.empty or len(df) < 20:
        return None

    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts").sort_index()

    close = df["close"].astype(float)

    # log returns
    returns = np.log(close / close.shift(1)).dropna()
    if returns.empty:
        return None

    # daily average return
    avg_daily_return = float(returns.mean())

    # annualized vol from 20-day rolling daily std
    rolling_vol = returns.rolling(20).std() * np.sqrt(252)
    avg_volatility = float(rolling_vol.dropna().mean()) if rolling_vol.notna().any() else np.nan

    # max drawdown on prices
    cum_max = close.cummax()
    drawdown = (close - cum_max) / cum_max
    max_drawdown = float(drawdown.min())

    # sharpe (annualized), assume rf=2%
    rf = 0.02
    annual_return = float(returns.mean() * 252)
    annual_vol = float(returns.std() * np.sqrt(252))
    sharpe_ratio = float((annual_return - rf) / annual_vol) if annual_vol > 0 else np.nan

    total_return = float((close.iloc[-1] - close.iloc[0]) / close.iloc[0])

    return {
        "ticker": ticker,
        "avg_daily_return": avg_daily_return,
        "avg_volatility": avg_volatility,
        "max_drawdown": max_drawdown,
        "sharpe_ratio": sharpe_ratio,
        "total_return": total_return,
    }


def portfolio_dashboard(conn: sqlite3.Connection, successful: list[str], window: DateWindow, output_dir: Path) -> pd.DataFrame:
    portfolio_rows = []

    for ticker in tqdm(successful, desc="Analyzing portfolio"):
        try:
            m = compute_metrics_for_ticker(conn, ticker, window)
            if m is None:
                continue
            m["report_path"] = str(output_dir / f"{ticker}_report.ipynb")
            portfolio_rows.append(m)
        except Exception as e:
            print(f"\nWarning: Could not calculate metrics for {ticker}: {e}")

    summary_df = pd.DataFrame(portfolio_rows)

    if summary_df.empty:
        print("No metrics computed (summary_df is empty).")
        return summary_df

    summary_df = summary_df.sort_values("sharpe_ratio", ascending=False)

    # Save summary
    summary_path = output_dir / "portfolio_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✓ Portfolio summary saved to: {summary_path}")

    # Show top/bottom 10
    def _pretty(df: pd.DataFrame) -> pd.DataFrame:
        out = df[["ticker", "total_return", "avg_volatility", "sharpe_ratio"]].copy()
        out["total_return"] *= 100
        out["avg_volatility"] *= 100
        return out

    print("\n" + "=" * 60)
    print("TOP 10 STOCKS BY SHARPE RATIO")
    print("=" * 60)
    print(_pretty(summary_df.head(10)).to_string(index=False))

    print("\n" + "=" * 60)
    print("BOTTOM 10 STOCKS BY SHARPE RATIO")
    print("=" * 60)
    print(_pretty(summary_df.tail(10)).to_string(index=False))

    # Plot risk-return
    plot_path = output_dir / "portfolio_risk_return.png"
    make_risk_return_plot(summary_df, plot_path)
    print(f"✓ Portfolio visualization saved to: {plot_path}")

    return summary_df


def make_risk_return_plot(summary_df: pd.DataFrame, plot_path: Path) -> None:
    df = summary_df.dropna(subset=["avg_volatility", "total_return", "sharpe_ratio"]).copy()
    if df.empty:
        print("Skipping plot: no valid rows after dropping NaNs.")
        return

    plt.figure(figsize=(14, 10))

    scatter = plt.scatter(
        df["avg_volatility"] * 100,
        df["total_return"] * 100,
        c=df["sharpe_ratio"],
        s=100,
        alpha=0.6,
        cmap="RdYlGn",
        edgecolors="black",
        linewidth=1,
    )

    cbar = plt.colorbar(scatter)
    cbar.set_label("Sharpe Ratio", fontsize=12)

    # Label top 5 by total return
    top_5 = df.nlargest(5, "total_return")
    for _, row in top_5.iterrows():
        plt.annotate(
            row["ticker"],
            xy=(row["avg_volatility"] * 100, row["total_return"] * 100),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
        )

    plt.xlabel("Average Volatility (%)", fontsize=12)
    plt.ylabel("Total Return (%)", fontsize=12)
    plt.title("Risk-Return Profile of Portfolio", fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()


In [15]:


# ----------------------------
# Main
# ----------------------------
def main():
    output_dir = ensure_output_dir()

    with sqlite3.connect(DB_PATH) as conn:
        tickers = get_tickers(conn)
        if not tickers:
            print("No tickers found. Exiting.")
            return

        window = get_last_6_complete_months(conn)
        print(f"Tickers: {len(tickers)}")
        print(f"Date range: {window.start_str} to {window.end_str} (last 6 complete months)")
        print(f"Output directory: {output_dir}")

        print("\n" + "=" * 60)
        print("Generating reports for all tickers...")
        print("=" * 60)

        successful, failed = run_reports(tickers, window, output_dir, TEMPLATE_PATH)

        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"Total tickers: {len(tickers)}")
        print(f"Successful: {len(successful)}")
        print(f"Failed: {len(failed)}")

        if failed:
            failed_path = output_dir / "failed_stocks.txt"
            with open(failed_path, "w", encoding="utf-8") as f:
                for ticker, tb in failed:
                    # first line is enough for the txt file; full tb in papermill_failures.log
                    first_line = tb.strip().splitlines()[-1] if tb else "Unknown error"
                    f.write(f"{ticker}: {first_line}\n")
            print(f"Failed stocks saved to: {failed_path}")
            print(f"Full tracebacks saved to: {output_dir / 'papermill_failures.log'}")

        if successful:
            portfolio_dashboard(conn, successful, window, output_dir)

    print("\n" + "=" * 60)
    print("ALL TASKS COMPLETE!")
    print("=" * 60)
    print(f"Reports saved to: {output_dir}")
    print("=" * 60)


if __name__ == "__main__":
    main()


Tickers: 30
Date range: 2024-06-01 to 2024-11-30 (last 6 complete months)
Output directory: reports\2026-03-05

Generating reports for all tickers...


Generating reports:   0%|                                                                                                                       | 0/30 [00:00<?, ?it/s]Passed unknown parameter: ticker
Passed unknown parameter: start_date
Passed unknown parameter: end_date
Input notebook does not contain a cell with tag 'parameters'


Executing:   0%|          | 0/10 [00:00<?, ?cell/s]

Generating reports:   3%|███▋                                                                                                           | 1/30 [00:12<06:05, 12.59s/it]Passed unknown parameter: ticker
Passed unknown parameter: start_date
Passed unknown parameter: end_date
Input notebook does not contain a cell with tag 'parameters'


Executing:   0%|          | 0/10 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/10 [00:00<?, ?cell/s]

Generating reports: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [04:59<00:00,  9.99s/it]



SUMMARY
Total tickers: 30
Successful: 30
Failed: 0


Analyzing portfolio: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:04<00:00,  7.06it/s]



✓ Portfolio summary saved to: reports\2026-03-05\portfolio_summary.csv

TOP 10 STOCKS BY SHARPE RATIO
ticker  total_return  avg_volatility  sharpe_ratio
   IBM     37.590755       19.325444      2.920560
    HD     26.520533       19.983199      2.831472
   SHW     28.583253       19.055711      2.775589
   WMT     23.533880       17.385506      2.749724
   MCD     21.863330       16.124623      2.656337
  CSCO     21.629153       19.200715      2.279256
   TRV     26.205192       23.047791      2.067278
   MMM     35.465116       30.286702      2.034036
   CRM     23.801876       25.303751      1.957144
  AAPL     21.115291       22.335433      1.769804

BOTTOM 10 STOCKS BY SHARPE RATIO
ticker  total_return  avg_volatility  sharpe_ratio
     V      7.485761       20.015301      0.824791
    VZ      7.369449       19.758740      0.764263
  NVDA     20.000000       58.963009      0.749570
  AMGN      4.632099       20.591202      0.436664
   CVX      2.838816       19.529997      0.178